In [ ]:
import os
import torch

# ==========================================
# 1. إعدادات الحساب والمستودع
# ==========================================
GITHUB_TOKEN = ""  # ضع التوكين هنا عند التشغيل في Colab فقط
GITHUB_USERNAME = "ApexBuildsHQ"
REPO_NAME = "Chief-King"
USER_EMAIL = ""    # ضع بريدك الإلكتروني هنا عند التشغيل في Colab

print("1. Checking GPU availability...")
if torch.cuda.is_available():
    print(f"--> GPU Active: {torch.cuda.get_device_name(0)}")
else:
    print("--> WARNING: GPU not active! Enable T4 GPU in Runtime settings.")

# ==========================================
# 2. سحب المستودع من GitHub
# ==========================================
print("\n2. Cloning Repository...")
!git config --global user.email "{USER_EMAIL}"
!git config --global user.name "{GITHUB_USERNAME}"
!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git

%cd {REPO_NAME}

# ==========================================
# 3. تثبيت المكتبات المطلوبة
# ==========================================
print("\n3. Installing Dependencies...")
!pip install -q transformers sentencepiece torch

# ==========================================
# 4. تعديل السكربت ليعمل على كارت الشاشة GPU
# ==========================================
print("\n4. Patching translate script for GPU support...")
with open("scripts/translate_chunk.py", "r", encoding="utf-8") as f:
    code = f.read()

if 'device = "cuda"' not in code:
    code = code.replace(
        'model = AutoModelForSeq2SeqLM.from_pretrained(model_name)',
        'device = "cuda" if torch.cuda.is_available() else "cpu"\n    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)'
    )
    code = code.replace(
        'inputs = tokenizer(safe_batch, return_tensors="pt", padding=True, truncation=True, max_length=256)',
        'inputs = tokenizer(safe_batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)'
    )
    with open("scripts/translate_chunk.py", "w", encoding="utf-8") as f:
        f.write(code)

# ==========================================
# 5. تشغيل الترجمة للـ 20 Chunk
# ==========================================
print("\n5. Running Translation Matrix (Chunks 1 to 20)...\n")
for chunk_id in range(1, 21):
    print(f"\n================ Processing Chunk {chunk_id}/20 ================")
    !python scripts/translate_chunk.py --chunk-id {chunk_id}

# ==========================================
# 6. دمج البيانات وبناء الفهارس
# ==========================================
print("\n6. Building Global Assets & Indexes...")
!python scripts/build_global_assets.py

# ==========================================
# 7. رفع النتائج المكتملة إلى GitHub
# ==========================================
print("\n7. Pushing all results to GitHub...")
!git add public/data/
!git commit -m "Auto-generated 50-language chunks via Google Colab T4 GPU [skip ci]"
!git push https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git main

print("\n🎉 ALL TASKS COMPLETED SUCCESSFULLY!")